In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("vehicles_dataset.csv")
df.head()

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X = df.drop(columns=['prezzo', 'price'], errors='ignore') # Assicurati del nome corretto della colonna target
y = df['price'] # o 'prezzo'

# Divisione
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 1. RIMOZIONE VALORI NULLI ---
colonne_da_droppare = ['fuel', 'year', 'model', 'transmission', 'title_status', 'odometer', 'manufacturer']

X_train = X_train.dropna(subset=colonne_da_droppare)
y_train = y_train.loc[X_train.index] # Allineiamo le etichette (y) agli indici rimasti in X

X_test = X_test.dropna(subset=colonne_da_droppare)
y_test = y_test.loc[X_test.index]

# --- 2. RIEMPIMENTO VALORI NULLI CON 'Unknown' (Corretto) ---
# Attenzione: lo facciamo su X_train e X_test, non su df!
colonne_da_riempire = ['condition', 'cylinders', 'drive', 'size', 'type', 'paint_color']

X_train[colonne_da_riempire] = X_train[colonne_da_riempire].fillna('Unknown')
X_test[colonne_da_riempire] = X_test[colonne_da_riempire].fillna('Unknown')

print(f"Dimensioni finali X_train: {X_train.shape}")
print(f"Dimensioni finali X_test: {X_test.shape}")

# --- 3. SALVATAGGIO IN CSV ---
print("Salvataggio dei file CSV in corso...")

# index=False evita di salvare la numerazione delle righe come colonna aggiuntiva nel CSV
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

# y_train e y_test sono Series, ma il metodo to_csv funziona allo stesso modo
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Tutti e 4 i dataset sono stati salvati con successo!")

Dimensioni finali X_train: (311786, 14)
Dimensioni finali X_test: (77818, 14)
Salvataggio dei file CSV in corso...
Tutti e 4 i dataset sono stati salvati con successo!


# Random Forest 

IL Random Forest non accetta valori stringhe o categorie, quindi vanno trasformati tutti i valori prima di poter applicare il modello.

In [2]:
# trasformazione colonna per colonna 
# in ordine year, facciamo la differenza tra anno corrente e anno dell' auto e sovrascriviamo la colonna 
# In questo modo il numero parte da 0 a salire per valori interi e il modello dovrebbe capire che man mano che sale 

from datetime import datetime

# Otteniamo l'anno corrente in automatico
anno_corrente = datetime.now().year
print(anno_corrente)

# Creiamo la nuova feature "eta_auto" sottraendo l'anno di immatricolazione
X_train['eta_auto'] = (anno_corrente - X_train['year']).astype(int)
X_test['eta_auto'] = (anno_corrente - X_test['year']).astype(int)

# (Opzionale) Una volta calcolata l'età, l'anno originale potrebbe essere ridondante 
# per i modelli, quindi puoi decidere di rimuovere la colonna originale
X_train = X_train.drop(columns=['year'])
X_test = X_test.drop(columns=['year'])

print(X_train.head())

2026
       manufacturer                  model  condition    cylinders fuel  \
366318         ford                  f-150    Unknown  8 cylinders  gas   
56271         acura                    mdx    Unknown  6 cylinders  gas   
264620         ford       f-250 super duty  excellent  8 cylinders  gas   
341412        buick  enclave essence sport       good  6 cylinders  gas   
153889         ford        f150 fx/2 sport  excellent  8 cylinders  gas   

        odometer title_status transmission drive       size    type  \
366318   48544.0        clean    automatic   4wd    Unknown   truck   
56271    53103.0        clean    automatic   fwd    Unknown     SUV   
264620   73394.0        clean    automatic   4wd  full-size   truck   
341412   38459.0        clean        other   fwd    Unknown     SUV   
153889  173267.0        clean    automatic   rwd  full-size  pickup   

       paint_color state  eta_auto  
366318       black    tx         9  
56271      Unknown    ca         8  
264620

In [3]:
# Passiamo ai cilindri sono facili da convertire mettiamo solo i numeri dei cilindri, visualizziamo quali sono i valori unici. 
X_train['cylinders'].unique()
# label_encoder per trasfomrare le colonne in numeri 

array(['8 cylinders', '6 cylinders', 'Unknown', '4 cylinders',
       '5 cylinders', 'other', '3 cylinders', '10 cylinders',
       '12 cylinders'], dtype=object)

In [4]:
mappa_cilindri = {
    '8 cylinders': 8,
    '6 cylinders': 6,
    '4 cylinders': 4,
    '5 cylinders': 5,
    '3 cylinders': 3,
    '10 cylinders': 10,
    '12 cylinders': 12,
    'Unknown': 0,
    'other': 0
}

# Aggiungiamo .fillna(0) prima di .astype(int)
X_train['cylinders'] = X_train['cylinders'].map(mappa_cilindri).fillna(0).astype(int)

# Facciamo lo stesso identico passaggio su X_test
X_test['cylinders'] = X_test['cylinders'].map(mappa_cilindri).fillna(0).astype(int)

# Verifica finale per essere sicuri
print("Esecuzione riuscita! Nuovi valori unici:")
print(X_train['cylinders'])

Esecuzione riuscita! Nuovi valori unici:
366318    8
56271     6
264620    8
341412    6
153889    8
         ..
119879    0
259178    5
365838    0
146867    4
121958    8
Name: cylinders, Length: 311786, dtype: int64


In [5]:
print(X_train.head())
print(X_train['cylinders'].unique())

       manufacturer                  model  condition  cylinders fuel  \
366318         ford                  f-150    Unknown          8  gas   
56271         acura                    mdx    Unknown          6  gas   
264620         ford       f-250 super duty  excellent          8  gas   
341412        buick  enclave essence sport       good          6  gas   
153889         ford        f150 fx/2 sport  excellent          8  gas   

        odometer title_status transmission drive       size    type  \
366318   48544.0        clean    automatic   4wd    Unknown   truck   
56271    53103.0        clean    automatic   fwd    Unknown     SUV   
264620   73394.0        clean    automatic   4wd  full-size   truck   
341412   38459.0        clean        other   fwd    Unknown     SUV   
153889  173267.0        clean    automatic   rwd  full-size  pickup   

       paint_color state  eta_auto  
366318       black    tx         9  
56271      Unknown    ca         8  
264620       white    n

In [6]:
# Ora togliamo lo stato  
# Rimuoviamo la colonna 'state' da entrambi i dataset
X_train = X_train.drop(columns=['state'], errors='ignore')
X_test = X_test.drop(columns=['state'], errors='ignore')

print("Colonna 'state' rimossa con successo!")
print(X_train.head())

Colonna 'state' rimossa con successo!
       manufacturer                  model  condition  cylinders fuel  \
366318         ford                  f-150    Unknown          8  gas   
56271         acura                    mdx    Unknown          6  gas   
264620         ford       f-250 super duty  excellent          8  gas   
341412        buick  enclave essence sport       good          6  gas   
153889         ford        f150 fx/2 sport  excellent          8  gas   

        odometer title_status transmission drive       size    type  \
366318   48544.0        clean    automatic   4wd    Unknown   truck   
56271    53103.0        clean    automatic   fwd    Unknown     SUV   
264620   73394.0        clean    automatic   4wd  full-size   truck   
341412   38459.0        clean        other   fwd    Unknown     SUV   
153889  173267.0        clean    automatic   rwd  full-size  pickup   

       paint_color  eta_auto  
366318       black         9  
56271      Unknown         8  
264

In [7]:
print(X_train['size'].unique())

['Unknown' 'full-size' 'mid-size' 'compact' 'sub-compact']


In [8]:
# Ordiniamo le categorie in base alla grandezza reale del veicolo:
# 0 = Sconosciuto, 1 = Piccolissima, 2 = Compatta, 3 = Media, 4 = Grande

mappa_size = {
    'Unknown': 0,
    'sub-compact': 1,
    'compact': 2,
    'mid-size': 3,
    'full-size': 4
}

# 1. Conversione del set di Allenamento (X_train)
X_train['size'] = X_train['size'].map(mappa_size).fillna(0).astype(int)

# 2. Conversione del set di Test (X_test) per mantenere i dati allineati
X_test['size'] = X_test['size'].map(mappa_size).fillna(0).astype(int)

# 3. Verifica visiva del risultato finale
print("Conversione completata!")
print("Valori unici in X_train['size']:", X_train['size'].unique())
print("Tipo di dato colonna 'size':", X_train['size'].dtype)
# =========================================================================

Conversione completata!
Valori unici in X_train['size']: [0 4 3 2 1]
Tipo di dato colonna 'size': int64


In [9]:
print(X_train.head())

       manufacturer                  model  condition  cylinders fuel  \
366318         ford                  f-150    Unknown          8  gas   
56271         acura                    mdx    Unknown          6  gas   
264620         ford       f-250 super duty  excellent          8  gas   
341412        buick  enclave essence sport       good          6  gas   
153889         ford        f150 fx/2 sport  excellent          8  gas   

        odometer title_status transmission drive  size    type paint_color  \
366318   48544.0        clean    automatic   4wd     0   truck       black   
56271    53103.0        clean    automatic   fwd     0     SUV     Unknown   
264620   73394.0        clean    automatic   4wd     4   truck       white   
341412   38459.0        clean        other   fwd     0     SUV       white   
153889  173267.0        clean    automatic   rwd     4  pickup       black   

        eta_auto  
366318         9  
56271          8  
264620        10  
341412         8

In [10]:
print(X_train['type'].unique())
# qui farò one hot encodig in maniera intelligente 

['truck' 'SUV' 'pickup' 'sedan' 'Unknown' 'other' 'van' 'coupe'
 'convertible' 'hatchback' 'offroad' 'wagon' 'mini-van' 'bus']


In [11]:
# =========================================================================
# 1. RAGGRUPPAMENTO INTELLIGENTE DELLA COLONNA 'TYPE'
# =========================================================================
# Creiamo il dizionario per ridurre le 14 categorie originali in 6 macro-gruppi

mappa_macro_tipi = {
    'truck': 'truck_pickup', 'pickup': 'truck_pickup',
    'SUV': 'suv_offroad', 'offroad': 'suv_offroad',
    'sedan': 'sedan_wagon', 'wagon': 'sedan_wagon',
    'coupe': 'sport_compact', 'convertible': 'sport_compact', 'hatchback': 'sport_compact',
    'van': 'van_minivan', 'mini-van': 'van_minivan',
    'Unknown': 'other_unknown', 'other': 'other_unknown', 'bus': 'other_unknown'
}

# Applichiamo la mappatura a X_train e X_test
X_train['type'] = X_train['type'].map(mappa_macro_tipi).fillna('other_unknown')
X_test['type'] = X_test['type'].map(mappa_macro_tipi).fillna('other_unknown')


# =========================================================================
# 2. APPLICAZIONE DEL ONE-HOT ENCODING BINARIO
# =========================================================================
# pd.get_dummies trasforma la colonna 'type' nelle nuove colonne (0 o 1)
# Usiamo dtype=int per avere direttamente numeri interi (0 e 1) invece di True/False

X_train = pd.get_dummies(X_train, columns=['type'], prefix='type', dtype=int)
X_test = pd.get_dummies(X_test, columns=['type'], prefix='type', dtype=int)


# =========================================================================
# 3. ALLINEAMENTO STRUTTURALE TRA TRAIN E TEST
# =========================================================================
# Per sicurezza matematica, ci assicuriamo che le colonne di X_test 
# siano identiche e nello stesso identico ordine di quelle di X_train

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# Verifica visiva delle nuove colonne generate
print("Colonne create dopo il One-Hot Encoding:")
print([col for col in X_train.columns if col.startswith('type_')])

Colonne create dopo il One-Hot Encoding:
['type_other_unknown', 'type_sedan_wagon', 'type_sport_compact', 'type_suv_offroad', 'type_truck_pickup', 'type_van_minivan']


In [12]:
print(X_train.head())

       manufacturer                  model  condition  cylinders fuel  \
366318         ford                  f-150    Unknown          8  gas   
56271         acura                    mdx    Unknown          6  gas   
264620         ford       f-250 super duty  excellent          8  gas   
341412        buick  enclave essence sport       good          6  gas   
153889         ford        f150 fx/2 sport  excellent          8  gas   

        odometer title_status transmission drive  size paint_color  eta_auto  \
366318   48544.0        clean    automatic   4wd     0       black         9   
56271    53103.0        clean    automatic   fwd     0     Unknown         8   
264620   73394.0        clean    automatic   4wd     4       white        10   
341412   38459.0        clean        other   fwd     0       white         8   
153889  173267.0        clean    automatic   rwd     4       black        18   

        type_other_unknown  type_sedan_wagon  type_sport_compact  \
366318      

In [13]:
# Ora facciamo one hot encoding della marca
# Applichiamo il One-Hot Encoding a tutte le ~40 marche presenti nel dataset.

# 1. Applicazione del One-Hot Encoding su X_train e X_test
# Usiamo il prefisso 'make' per riconoscere facilmente queste colonne in futuro
X_train = pd.get_dummies(X_train, columns=['manufacturer'], prefix='make', dtype=int)
X_test = pd.get_dummies(X_test, columns=['manufacturer'], prefix='make', dtype=int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# 3. Verifica del numero di colonne create
colonne_marche = [col for col in X_train.columns if col.startswith('make_')]
print(f"One-Hot Encoding completato con successo!")
print(f"Numero totale di marche codificate come colonne: {len(colonne_marche)}")
# =========================================================================

One-Hot Encoding completato con successo!
Numero totale di marche codificate come colonne: 41


In [14]:
print(X_train.head())

                        model  condition  cylinders fuel  odometer  \
366318                  f-150    Unknown          8  gas   48544.0   
56271                     mdx    Unknown          6  gas   53103.0   
264620       f-250 super duty  excellent          8  gas   73394.0   
341412  enclave essence sport       good          6  gas   38459.0   
153889        f150 fx/2 sport  excellent          8  gas  173267.0   

       title_status transmission drive  size paint_color  ...  make_pontiac  \
366318        clean    automatic   4wd     0       black  ...             0   
56271         clean    automatic   fwd     0     Unknown  ...             0   
264620        clean    automatic   4wd     4       white  ...             0   
341412        clean        other   fwd     0       white  ...             0   
153889        clean    automatic   rwd     4       black  ...             0   

        make_porsche  make_ram  make_rover  make_saturn  make_subaru  \
366318             0         0  

In [15]:
print(X_train['condition'].unique())

['Unknown' 'excellent' 'good' 'like new' 'fair' 'salvage' 'new']


In [16]:
# Mappiamo lo stato di usura dell'auto con un punteggio numerico crescente da 0 a 6.
# Questo permette alla Random Forest di capire la progressione qualitativa.

mappa_condition = {
    'Unknown': 0,
    'salvage': 1,
    'fair': 2,
    'good': 3,
    'excellent': 4,
    'like new': 5,
    'new': 6
}

# 1. Conversione del set di Allenamento (X_train)
X_train['condition'] = X_train['condition'].map(mappa_condition).fillna(0).astype(int)

# 2. Conversione del set di Test (X_test) per mantenere i dati allineati
X_test['condition'] = X_test['condition'].map(mappa_condition).fillna(0).astype(int)

# 3. Verifica visiva del risultato finale
print("Conversione della colonna 'condition' completata!")
print("Valori unici in X_train['condition']:", X_train['condition'].unique())
print("Tipo di dato colonna 'condition':", X_train['condition'].dtype)
# =========================================================================

Conversione della colonna 'condition' completata!
Valori unici in X_train['condition']: [0 4 3 5 2 1 6]
Tipo di dato colonna 'condition': int64


In [17]:
print(X_train.head())

                        model  condition  cylinders fuel  odometer  \
366318                  f-150          0          8  gas   48544.0   
56271                     mdx          0          6  gas   53103.0   
264620       f-250 super duty          4          8  gas   73394.0   
341412  enclave essence sport          3          6  gas   38459.0   
153889        f150 fx/2 sport          4          8  gas  173267.0   

       title_status transmission drive  size paint_color  ...  make_pontiac  \
366318        clean    automatic   4wd     0       black  ...             0   
56271         clean    automatic   fwd     0     Unknown  ...             0   
264620        clean    automatic   4wd     4       white  ...             0   
341412        clean        other   fwd     0       white  ...             0   
153889        clean    automatic   rwd     4       black  ...             0   

        make_porsche  make_ram  make_rover  make_saturn  make_subaru  \
366318             0         0  

In [18]:
print(X_train['drive'].unique())

['4wd' 'fwd' 'rwd' 'Unknown']


In [19]:
# rimuoviamo la colonna 'drive_Unknown', i veicoli senza
# trazione specificata avranno automaticamente 0 in 4wd, fwd e rwd.

# 1. Applicazione del One-Hot Encoding standard su X_train e X_test
X_train = pd.get_dummies(X_train, columns=['drive'], prefix='drive', dtype=int)
X_test = pd.get_dummies(X_test, columns=['drive'], prefix='drive', dtype=int)


# 2. Rimozione della colonna 'drive_Unknown' da entrambi i dataset
# In questo modo, dove c'era 1 in Unknown, rimarranno solo gli 0 nelle altre colonne
X_train = X_train.drop(columns=['drive_Unknown'], errors='ignore')
X_test = X_test.drop(columns=['drive_Unknown'], errors='ignore')

# Ci assicuriamo che X_test segua l'esatta struttura di X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# 4. Verifica visiva delle colonne rimaste
colonne_trazione = [col for col in X_train.columns if col.startswith('drive_')]
print("Conversione della trazione completata con la logica degli zeri!")
print("Colonne effettive nel dataset:", colonne_trazione)
# =========================================================================

Conversione della trazione completata con la logica degli zeri!
Colonne effettive nel dataset: ['drive_4wd', 'drive_fwd', 'drive_rwd']


In [20]:
print(X_train.head())

                        model  condition  cylinders fuel  odometer  \
366318                  f-150          0          8  gas   48544.0   
56271                     mdx          0          6  gas   53103.0   
264620       f-250 super duty          4          8  gas   73394.0   
341412  enclave essence sport          3          6  gas   38459.0   
153889        f150 fx/2 sport          4          8  gas  173267.0   

       title_status transmission  size paint_color  eta_auto  ...  make_rover  \
366318        clean    automatic     0       black         9  ...           0   
56271         clean    automatic     0     Unknown         8  ...           0   
264620        clean    automatic     4       white        10  ...           0   
341412        clean        other     0       white         8  ...           0   
153889        clean    automatic     4       black        18  ...           0   

        make_saturn  make_subaru  make_tesla  make_toyota  make_volkswagen  \
366318        

In [21]:
print(X_train['title_status'].unique())

['clean' 'rebuilt' 'salvage' 'lien' 'missing' 'parts only']


In [22]:
# =========================================================================
# TRASFORMAZIONE DELLA COLONNA 'TITLE_STATUS' (VARIABILE ORDINALE)
# =========================================================================
# Mappiamo lo stato legale del veicolo con un punteggio numerico crescente.
# Più il titolo è "problematico", più il valore si abbassa.

mappa_title = {
    'Unknown': 0,
    'parts only': 1,
    'salvage': 2,
    'missing': 3,
    'rebuilt': 4,
    'lien': 5,
    'clean': 6
}

# 1. Conversione del set di Allenamento (X_train)
X_train['title_status'] = X_train['title_status'].map(mappa_title).fillna(0).astype(int)

# 2. Conversione del set di Test (X_test) per mantenere i dati allineati
X_test['title_status'] = X_test['title_status'].map(mappa_title).fillna(0).astype(int)

# 3. Verifica visiva del risultato finale
print("Conversione della colonna 'title_status' completata!")
print("Valori unici in X_train['title_status']:", X_train['title_status'].unique())
print("Tipo di dato colonna 'title_status':", X_train['title_status'].dtype)
# =========================================================================

Conversione della colonna 'title_status' completata!
Valori unici in X_train['title_status']: [6 4 2 5 3 1]
Tipo di dato colonna 'title_status': int64


In [23]:
print(X_train.head())

                        model  condition  cylinders fuel  odometer  \
366318                  f-150          0          8  gas   48544.0   
56271                     mdx          0          6  gas   53103.0   
264620       f-250 super duty          4          8  gas   73394.0   
341412  enclave essence sport          3          6  gas   38459.0   
153889        f150 fx/2 sport          4          8  gas  173267.0   

        title_status transmission  size paint_color  eta_auto  ...  \
366318             6    automatic     0       black         9  ...   
56271              6    automatic     0     Unknown         8  ...   
264620             6    automatic     4       white        10  ...   
341412             6        other     0       white         8  ...   
153889             6    automatic     4       black        18  ...   

        make_rover  make_saturn  make_subaru  make_tesla  make_toyota  \
366318           0            0            0           0            0   
56271       

In [24]:
# =========================================================================
# CONVERSIONE DELLA COLONNA 'ODOMETER' (DA MIGLIA A CHILOMETRI)
# =========================================================================
# Moltiplichiamo il valore originario in miglia per il fattore 1.60934.
# Convertiamo il risultato in numero intero (int) per eliminare i decimali.

# 1. Conversione del set di Allenamento (X_train)
X_train['odometer'] = (X_train['odometer'] * 1.60934).astype(int)

# 2. Conversione del set di Test (X_test) per mantenere i dati allineati
X_test['odometer'] = (X_test['odometer'] * 1.60934).astype(int)

# 3. Verifica visiva delle prime righe convertite
print("Conversione da miglia a chilometri completata!")
print(X_train['odometer'].head())
# =========================================================================

Conversione da miglia a chilometri completata!
366318     78123
56271      85460
264620    118115
341412     61893
153889    278845
Name: odometer, dtype: int64


In [25]:
print(X_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 311786 entries, 366318 to 121958
Data columns (total 60 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   model                 311786 non-null  object
 1   condition             311786 non-null  int64 
 2   cylinders             311786 non-null  int64 
 3   fuel                  311786 non-null  object
 4   odometer              311786 non-null  int64 
 5   title_status          311786 non-null  int64 
 6   transmission          311786 non-null  object
 7   size                  311786 non-null  int64 
 8   paint_color           311786 non-null  object
 9   eta_auto              311786 non-null  int64 
 10  type_other_unknown    311786 non-null  int64 
 11  type_sedan_wagon      311786 non-null  int64 
 12  type_sport_compact    311786 non-null  int64 
 13  type_suv_offroad      311786 non-null  int64 
 14  type_truck_pickup     311786 non-null  int64 
 15  type_van_minivan 

In [27]:
# =========================================================================
# RIMOZIONE DELLA COLONNA 'PAINT_COLOR'
# =========================================================================
X_train = X_train.drop(columns=['paint_color'], errors='ignore')
X_test = X_test.drop(columns=['paint_color'], errors='ignore')
print("Colonna 'paint_color' eliminata dal dataset.")
# =========================================================================

Colonna 'paint_color' eliminata dal dataset.


In [26]:
print(X_train['transmission'].unique())

['automatic' 'other' 'manual']


In [28]:
# =========================================================================
# ONE-HOT ENCODING DELLA COLONNA 'TRANSMISSION' (TIPO DI CAMBIO)
# =========================================================================
# Applichiamo il One-Hot Encoding completo mantenendo tutte e tre le categorie.
# Come hai giustamente notato, 'other' rappresenta tipi di cambio reali 
# (es. CVT o sequenziali) e merita una sua colonna numerica dedicata.

# 1. Applicazione del One-Hot Encoding su X_train e X_test
# Usiamo il prefisso 'trans' per identificare le nuove colonne
X_train = pd.get_dummies(X_train, columns=['transmission'], prefix='trans', dtype=int)
X_test = pd.get_dummies(X_test, columns=['transmission'], prefix='trans', dtype=int)


# =========================================================================
# 2. ALLINEAMENTO STRUTTURALE TRA TRAIN E TEST
# =========================================================================
# Garantiamo che X_test abbia lo stesso identico ordine di colonne di X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# 3. Verifica visiva del risultato
colonne_cambio = [col for col in X_train.columns if col.startswith('trans_')]
print("Conversione della trasmissione completata!")
print("Colonne effettive nel dataset:", colonne_cambio)
# =========================================================================

Conversione della trasmissione completata!
Colonne effettive nel dataset: ['trans_automatic', 'trans_manual', 'trans_other']


In [29]:
print(X_train['fuel'].unique())

['gas' 'other' 'diesel' 'hybrid' 'electric']


In [30]:
# =========================================================================
# ONE-HOT ENCODING DELLA COLONNA 'FUEL' (TIPO DI ALIMENTAZIONE)
# =========================================================================
# Applichiamo il One-Hot Encoding completo mantenendo tutte le categorie.
# Come per la trasmissione, 'other' identifica alimentazioni alternative reali 
# (es. GPL o metano) e merita una sua colonna numerica dedicata.

# 1. Applicazione del One-Hot Encoding su X_train e X_test
# Usiamo il prefisso 'fuel' per identificare le nuove colonne
X_train = pd.get_dummies(X_train, columns=['fuel'], prefix='fuel', dtype=int)
X_test = pd.get_dummies(X_test, columns=['fuel'], prefix='fuel', dtype=int)


# =========================================================================
# 2. ALLINEAMENTO STRUTTURALE TRA TRAIN E TEST
# =========================================================================
# Ci assicuriamo che X_test segua l'esatta struttura e ordine di X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# 3. Verifica visiva del risultato finale
colonne_carburante = [col for col in X_train.columns if col.startswith('fuel_')]
print("Conversione del carburante completata!")
print("Colonne effettive nel dataset:", colonne_carburante)
# =========================================================================

Conversione del carburante completata!
Colonne effettive nel dataset: ['fuel_diesel', 'fuel_electric', 'fuel_gas', 'fuel_hybrid', 'fuel_other']


In [31]:
print(X_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 311786 entries, 366318 to 121958
Data columns (total 65 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   model                 311786 non-null  object
 1   condition             311786 non-null  int64 
 2   cylinders             311786 non-null  int64 
 3   odometer              311786 non-null  int64 
 4   title_status          311786 non-null  int64 
 5   size                  311786 non-null  int64 
 6   eta_auto              311786 non-null  int64 
 7   type_other_unknown    311786 non-null  int64 
 8   type_sedan_wagon      311786 non-null  int64 
 9   type_sport_compact    311786 non-null  int64 
 10  type_suv_offroad      311786 non-null  int64 
 11  type_truck_pickup     311786 non-null  int64 
 12  type_van_minivan      311786 non-null  int64 
 13  make_acura            311786 non-null  int64 
 14  make_alfa-romeo       311786 non-null  int64 
 15  make_aston-martin

In [33]:
# togliamo gli outliers dal prezzo 
# Teniamo solo le auto con un prezzo reale e sensato (es. tra 500$ e 100.000$)
# Questo eliminerà sia i prezzi a 0/1$ sia i prezzi fake da milioni di dollari.

print(f"Righe iniziali Train: {X_train.shape[0]}")

# 1. Filtro per il set di allenamento (Train)
limite_prezzo_train = (y_train >= 500) & (y_train <= 100000)
X_train = X_train[limite_prezzo_train]
y_train = y_train[limite_prezzo_train]

# 2. Filtro per il set di test (Test)
limite_prezzo_test = (y_test >= 500) & (y_test <= 100000)
X_test = X_test[limite_prezzo_test]
y_test = y_test[limite_prezzo_test]

print(f"Righe rimaste Train dopo il filtro prezzo: {X_train.shape[0]}")
print(f"Righe rimaste Test dopo il filtro prezzo: {X_test.shape[0]}")
# =========================================================================

Righe iniziali Train: 311786
Righe rimaste Train dopo il filtro prezzo: 282489
Righe rimaste Test dopo il filtro prezzo: 70553


In [34]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================================
# 1. ISOLAMENTO DELLE SOLO COLONNE NUMERICHE (ESCLUSIONE DI 'MODEL')
# =========================================================================
# Questo passaggio elimina 'model' e pulisce il dataset da ogni testo residuo

X_train_rf = X_train.select_dtypes(include=[np.number])
X_test_rf = X_test.select_dtypes(include=[np.number])

print(f"Numero di feature (colonne) usate per l'addestramento: {X_train_rf.shape[1]}")

# 2. ADDESTRAMENTO DELLA RANDOM FOREST


# n_jobs=-1 usa tutta la potenza del tuo processore per accelerare i calcoli
modello_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modello_rf.fit(X_train_rf, y_train)

print("-> Addestramento completato con successo!")


# =========================================================================
# 3. GENERAZIONE PREDIZIONI E VALUTAZIONE METRICHE
# =========================================================================
y_pred = modello_rf.predict(X_test_rf)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n=========================================================================")
print("📊 NUOVE PRESTAZIONI DELLA RANDOM FOREST (SENZA COLONNA 'MODEL'):")
print(f"-> Errore Medio Assoluto (MAE): {mae:.2f} $")
print(f"-> Radice dell'Errore Quadratico Medio (RMSE): {rmse:.2f} $")
print(f"-> Coefficiente di Determinazione (R²): {r2:.4f}")
print("=========================================================================")

Numero di feature (colonne) usate per l'addestramento: 64
-> Addestramento completato con successo!

📊 NUOVE PRESTAZIONI DELLA RANDOM FOREST (SENZA COLONNA 'MODEL'):
-> Errore Medio Assoluto (MAE): 1996.95 $
-> Radice dell'Errore Quadratico Medio (RMSE): 4514.66 $
-> Coefficiente di Determinazione (R²): 0.8991


In [16]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Carica i file originali
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

# 2. Creiamo dei NUOVI dataframe copiando gli originali per non sovrascriverli
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# 3. Definiamo le colonne testuali
colonne_testuali = ['manufacturer', 'model', 'condition', 'cylinders', 
                    'fuel', 'title_status', 'transmission', 'drive', 
                    'size', 'type', 'paint_color', 'state']

# 4. Inizializziamo l'encoder
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# 5. Applichiamo la trasformazione sui NUOVI dataframe
X_train_encoded[colonne_testuali] = encoder.fit_transform(X_train[colonne_testuali])
X_test_encoded[colonne_testuali] = encoder.transform(X_test[colonne_testuali])

# --- GENERAZIONE DELLA LEGENDA IN UN CSV ---
print("Generazione della legenda delle conversioni...")

righe_legenda = []

# Iteriamo su ogni colonna per estrarre la corrispondenza Testo -> Numero
for i, colonna in enumerate(colonne_testuali):
    # encoder.categories_[i] contiene la lista ordinata dei testi per quella colonna
    for numero, testo in enumerate(encoder.categories_[i]):
        righe_legenda.append({
            'colonna': colonna,
            'valore_originale': testo,
            'valore_numerico': numero
        })

# Trasformiamo la lista in un comodo DataFrame di Pandas
df_legenda = pd.DataFrame(righe_legenda)

# Salviamo la legenda in un nuovo CSV
df_legenda.to_csv('legenda_encoding.csv', index=False)
print("-> File 'legenda_encoding.csv' creato con successo!")

# (Opzionale) Se vuoi, puoi salvare anche i nuovi dataset codificati
X_train_encoded.to_csv('X_train_encoded.csv', index=False)
X_test_encoded.to_csv('X_test_encoded.csv', index=False)
print("-> Nuovi dataset codificati salvati!")

Generazione della legenda delle conversioni...
-> File 'legenda_encoding.csv' creato con successo!
-> Nuovi dataset codificati salvati!


In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("1. Caricamento dei dataset codificati...")
# Carichiamo i dati numerici che hai salvato nel passaggio precedente
X_train = pd.read_csv('X_train_encoded.csv')
#X_test = pd.read_csv('X_test.csv') # Usiamo quello non codificato se l'encoder lo applichi al volo, oppure X_test_encoded.csv
# NOTA: Assicurati di usare X_test_encoded.csv che abbiamo creato prima!
X_test = pd.read_csv('X_test_encoded.csv')

y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

print(f"   -> Dati pronti per il training: {X_train.shape[0]} righe.")

print("\n2. Configurazione del modello Random Forest...")
# Inizializziamo l'algoritmo
# n_estimators=100 significa che il modello creerà 100 alberi decisionali in parallelo
# n_jobs=-1 dice a Python di usare tutti i core del tuo processore per fare i calcoli più velocemente
modello_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

print("3. Inizio addestramento (fit)... Questa operazione potrebbe richiedere qualche minuto...")
modello_rf.fit(X_train, y_train)
print("   -> Addestramento completato!")

print("\n4. Generazione delle predizioni sul set di test...")
# Chiediamo al modello di stimare i prezzi per le auto che non ha mai visto (X_test)
y_pred = modello_rf.predict(X_test)

print("\n5. VALUTAZIONE DELLE PRESTAZIONI:")
# Calcoliamo le metriche per capire se il modello è preciso

# MAE: ci dice di quanti dollari (o euro) il modello si sbaglia in media
mae = mean_absolute_error(y_test, y_pred)
# RMSE: simile al MAE, ma penalizza molto di più gli errori grandi
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# R² (R-quadrato): va da 0 a 1. Più è vicino a 1, più il modello è accurato
r2 = r2_score(y_test, y_pred)

print(f"   -> Errore Medio Assoluto (MAE): {mae:.2f} $")
print(f"   -> Radice dell'Errore Quadratico Medio (RMSE): {rmse:.2f} $")
print(f"   -> Coefficiente di Determinazione (R²): {r2:.4f}")

print("\n6. Salvataggio del modello su file...")
# Salviamo il modello addestrato in modo che sia pronto per l'Ensemble finale del team
joblib.dump(modello_rf, 'modello_random_forest.joblib')
print("   -> File 'modello_random_forest.joblib' creato con successo!")

1. Caricamento dei dataset codificati...
   -> Dati pronti per il training: 311786 righe.

2. Configurazione del modello Random Forest...
3. Inizio addestramento (fit)... Questa operazione potrebbe richiedere qualche minuto...
   -> Addestramento completato!

4. Generazione delle predizioni sul set di test...

5. VALUTAZIONE DELLE PRESTAZIONI:
   -> Errore Medio Assoluto (MAE): 151704.48 $
   -> Radice dell'Errore Quadratico Medio (RMSE): 18447902.74 $
   -> Coefficiente di Determinazione (R²): -0.1450

6. Salvataggio del modello su file...
   -> File 'modello_random_forest.joblib' creato con successo!
